# Risk scoring without hypothetical loans

All 7,500 applications use the request-time classifier and all 5,000 customers use their actual profile classifier. The customer estimate describes latest-observed-loan label association, not the risk of a particular future loan.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
OUT = ROOT / 'artifacts'
assert json.loads((OUT / 'run_manifest.json').read_text())['version'] == 2, 'Run python -m src.pipeline first'

from src.risk_score import score
display(score([0,.33,.34,.66,.67,1]))
apps=pd.read_csv(OUT/'scored_applications.csv')
customers=pd.read_csv(OUT/'scored_customers.csv')
assert len(apps)==7500 and len(customers)==5000
assert customers.score_basis.eq('customer_profile_model').all()
assert customers.source_application_id.isna().all()
print('Hypothetical applications:',0)

## Exact bands and separate review thresholds
Round calibrated probability ×100 to an integer. Low is 0–33, Medium 34–66 and High 67–100. The classifier review threshold is selected on training folds and is separate from these fixed risk bands. Never label customers without outcomes as non-defaults.

In [ ]:
display(customers.groupby(['risk_category','score_basis']).size().rename('customers').reset_index())
customers.risk_category.value_counts().plot.pie(autopct='%1.1f%%',figsize=(6,6));plt.ylabel('');plt.title('Actual-profile customer risk distribution');plt.show()

## Reload both model artefacts
The feature contracts and context guard prevent a retrospective model from being silently used for new request scoring. Saved models remain fitted on training partitions, so reported holdout metrics remain distinct from in-sample dashboard scores.

In [ ]:
import joblib
from src.score_requests import score_requests
from src.data_load import read_raw
frames=read_raw();artifact=joblib.load(ROOT/'models/default_model.pkl')
requests=score_requests(frames['loan_applications'].head(30),frames['transactions'],artifact)
expected=apps.loc[apps.application_id.isin(requests.application_id)]
assert np.allclose(requests.default_probability,expected.default_probability)
print('Batch request scoring reproduces persisted probabilities.')

## Geographic watchlist and complete traces
The dashboard filters customer risk categories and regions, shows actual-profile drivers, and provides a separate applicant selector. Data for borrowers without outcomes remain extrapolations, but no application is fabricated.

In [ ]:
watch=pd.read_csv(OUT/'dashboard_customers.csv')
display(watch.groupby(['state','risk_category']).size().unstack(fill_value=0))
print('Applicant explanation rows:',len(json.loads((OUT/'prediction_explanations.json').read_text())))
print('Customer explanation rows:',len(json.loads((OUT/'customer_model/prediction_explanations.json').read_text())))

## Handover and remaining acceptance
Run streamlit run dashboard/app.py. Review all eight required components, both score populations and objective acceptance states. Historical bureau/profile snapshots and dated maturity outcomes are still required for true prospective validation; AUC is reported as measured rather than forced to meet the target.

In [ ]:
print((ROOT/'documentation/results.md').read_text())
print(json.loads((OUT/'run_manifest.json').read_text()))